# AQI Predictor — Exploratory Data Analysis
## Hyderabad, Pakistan — Air Quality Forecasting

This notebook explores the merged hourly dataset to understand:
- AQI trends and seasonality
- Weather-AQI correlations
- Optimal lag windows
- Missing data patterns

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

In [2]:
# Load merged hourly data
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    print(f'Loaded {len(df)} rows')
except FileNotFoundError:
    print('No merged data found. Run ingestion first.')
    df = pd.DataFrame()

No merged data found. Run ingestion first.


In [ ]:
if not df.empty:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').sort_index()
    print(f'Date range: {df.index.min()} → {df.index.max()}')
    print(f'Columns: {list(df.columns)}')
    display(df.describe())

## 1. AQI Time Series

In [ ]:
aqi_col = 'aqi' if 'aqi' in df.columns else 'us_aqi'
if not df.empty and aqi_col in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # Full time series
    df[aqi_col].plot(ax=axes[0], title='AQI Over Time', color='#00e400', linewidth=0.5)
    axes[0].set_ylabel('AQI')
    
    # Last 7 days
    last_7d = df[aqi_col].last('7D')
    last_7d.plot(ax=axes[1], title='Last 7 Days AQI', color='#ff7e00', linewidth=1.5, marker='o')
    axes[1].set_ylabel('AQI')
    
    plt.tight_layout()
    plt.show()

## 2. Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

if not df.empty and aqi_col in df.columns:
    # Resample to daily for decomposition
    daily = df[aqi_col].resample('D').mean().dropna()
    
    if len(daily) >= 14:
        decomp = seasonal_decompose(daily, model='additive', period=7)
        fig = decomp.plot()
        fig.set_size_inches(16, 12)
        plt.show()

## 3. Weather vs AQI Correlation

In [ ]:
weather_cols = ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 
                'pressure_msl', 'wind_speed_10m', 'precipitation', 'cloud_cover']
available_weather = [c for c in weather_cols if c in df.columns]

if not df.empty and aqi_col in df.columns and available_weather:
    corr_cols = [aqi_col] + available_weather
    corr = df[corr_cols].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('AQI vs Weather Correlation Matrix')
    plt.show()

## 4. AQI by Hour of Day

In [ ]:
if not df.empty and aqi_col in df.columns:
    df['hour'] = df.index.hour
    hourly_avg = df.groupby('hour')[aqi_col].mean()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    hourly_avg.plot(kind='bar', ax=ax, color='#00b4d8')
    ax.set_title('Average AQI by Hour of Day')
    ax.set_xlabel('Hour')
    ax.set_ylabel('Average AQI')
    plt.show()

## 5. Missing Data Analysis

In [ ]:
if not df.empty:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
    missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
    
    if not missing_df.empty:
        display(missing_df)
    else:
        print('No missing values found!')
    
    # Hourly continuity
    expected_hours = pd.date_range(df.index.min(), df.index.max(), freq='h')
    actual_hours = len(df)
    completeness = (actual_hours / len(expected_hours)) * 100
    print(f'\nHourly completeness: {completeness:.1f}% ({actual_hours}/{len(expected_hours)} hours)')

## 6. Lag Correlation Analysis

In [ ]:
if not df.empty and aqi_col in df.columns:
    max_lag = 96  # 4 days
    lags = range(1, max_lag + 1)
    autocorr = [df[aqi_col].autocorr(lag=lag) for lag in lags]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(lags, autocorr, color='#00b4d8', alpha=0.7)
    ax.axhline(y=0, color='white', linestyle='--', linewidth=0.5)
    ax.axvline(x=24, color='#ff7e00', linestyle='--', linewidth=1, label='24h')
    ax.axvline(x=48, color='#ff0000', linestyle='--', linewidth=1, label='48h')
    ax.axvline(x=72, color='#8f3f97', linestyle='--', linewidth=1, label='72h')
    ax.set_title('AQI Autocorrelation by Lag (hours)')
    ax.set_xlabel('Lag (hours)')
    ax.set_ylabel('Autocorrelation')
    ax.legend()
    plt.show()

## Summary

Key findings from EDA:
1. ...
2. ...
3. ...